# tau3 Multi-objective vs Single-objective GRPO (Colab Free T4)

Runs the pure-RL loop with **Qwen2.5-3B-Instruct** (QLoRA, custom agentic GRPO)
on the synthesized tau3 retail suite, for two arms:

- `single` — reward = correctness only
- `multi` — reward = weighted sum (correctness 1.0, efficiency 0.3, tool_safety 0.5)

Select **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

# project location
PROJECT = '/content/agentic'
# move checkpoints off the ephemeral disk so they survive session timeouts
os.environ.setdefault('CKPT', '/content/drive/MyDrive/agentic_tau3_rl')
print('Drive mounted, CKPT =', os.environ['CKPT'])

In [ ]:
# install uv, clone repo, sync deps
!pip -q install uv
if not os.path.isdir(PROJECT):
    !git clone https://github.com/culey24/agentic.git {PROJECT}
%cd {PROJECT}
!uv sync --dev
print('repo ready')

In [ ]:
# GPU stack: torch + unsloth (provides fast 4-bit LoRA + GRPO building blocks)
!uv pip install torch --index-url https://download.pytorch.org/whl/cu121
!uv pip install unsloth xformers trl peft bitsandbytes
print('GPU stack ready')

In [ ]:
# sanity: model loads + a dry rollout before spending session time on training
from experiments.rl.colab.run import _load_dotenv
_load_dotenv()
from experiments.rl.colab.local_provider import LocalQwenProvider
from experiments.rl.colab.rollout import Tau3Rollout, single_objective_reward
from experiments.multiobj.scorers.tau3 import make_tau3_scorer
from harnessx.benchmarks.tau3 import Tau3Adapter
from harnessx.benchmarks.tau3.retail import RetailDomain
import asyncio

async def dry():
    prov = LocalQwenProvider('Qwen/Qwen2.5-3B-Instruct')
    tasks = Tau3Adapter('examples/data/tau3_colab.jsonl').load_tasks()
    rec = await Tau3Rollout(prov, RetailDomain(), max_turns=20).run(
        tasks[0], make_tau3_scorer(max_turns=20), single_objective_reward)
    print('tool_calls:', [t.tool_calls for t in rec.turns])
    print('rewards:', rec.rewards, 'scalar:', rec.reward)

await dry()

In [ ]:
# Arm 1: SINGLE objective (correctness only)
ckpt = os.path.join(os.environ['CKPT'], 'single')
!uv run python experiments/rl/colab/run.py \
    --arm single \
    --rounds 6 --rollouts 8 --concurrency 4 --max-turns 200 \
    --lr 5e-5 --kl-beta 0.04 --clip-ratio 0.2 \
    --out {ckpt} --save-lora {ckpt}/lora \
    --model Qwen/Qwen2.5-3B-Instruct 2>&1 | tee {ckpt}.log

In [ ]:
# Arm 2: MULTI objective (weighted sum)
ckpt = os.path.join(os.environ['CKPT'], 'multi')
!uv run python experiments/rl/colab/run.py \
    --arm multi --weights correctness=1.0,efficiency=0.3,tool_safety=0.5 \
    --rounds 6 --rollouts 8 --concurrency 4 --max-turns 200 \
    --lr 5e-5 --kl-beta 0.04 --clip-ratio 0.2 \
    --out {ckpt} --save-lora {ckpt}/lora \
    --model Qwen/Qwen2.5-3B-Instruct 2>&1 | tee {ckpt}.log

In [ ]:
# Compare the two arms: pass@N curves + objective means + Pareto
import json, os, glob
import matplotlib.pyplot as plt

def load_rounds(arm):
    p = os.path.join(os.environ['CKPT'], arm, 'rounds.jsonl')
    return [json.loads(l) for l in open(p) if l.strip()] if os.path.exists(p) else []

single, multi = load_rounds('single'), load_rounds('multi')
print(f"{'round':<6}{'single pass@N':<16}{'multi pass@N':<16}{'single reward':<16}{'multi reward':<16}")
for i in range(max(len(single), len(multi))):
    s = single[i] if i < len(single) else {}
    m = multi[i] if i < len(multi) else {}
    print(f"R{i:<5}{s.get('pass_rate', float('nan')):<16.3f}{m.get('pass_rate', float('nan')):<16.3f}"
          f"{s.get('mean_reward', float('nan')):<16.3f}{m.get('mean_reward', float('nan')):<16.3f}")

plt.figure(figsize=(6, 4))
plt.plot([r['pass_rate'] for r in single], '-o', label='single')
plt.plot([r['pass_rate'] for r in multi], '-s', label='multi')
plt.xlabel('round'); plt.ylabel('pass@N'); plt.title('tau3 GRPO: single vs multi objective')
plt.legend(); plt.grid(alpha=.3); plt.show()

def means(arm):
    p = os.path.join(os.environ['CKPT'], arm, 'rollouts.jsonl')
    vecs = [json.loads(l)['rewards'] for l in open(p) if l.strip()] if os.path.exists(p) else []
    names = sorted({n for v in vecs for n in v})
    return {n: sum(v.get(n, 0) for v in vecs)/len(vecs) for n in names} if vecs else {}

print('objective means  single:', {k: round(v, 3) for k, v in means('single').items()})
print('objective means  multi :', {k: round(v, 3) for k, v in means('multi').items()})

### Interpretation
- If `multi` keeps `correctness` ≈ `single` while raising `efficiency`/`tool_safety`, the
  multi-objective reward shaping is **free**: it improves secondary objectives without
  sacrificing the primary one (verify no correctness regression in pass@N).
- If `single` regresses `efficiency`/`tool_safety` to win correctness, that is the expected
  single-objective overfit — evidence for why multi-objective matters.
- Checkpoints live under your Drive `CKPT` dir (`rollouts.jsonl`, `rounds.jsonl`, `lora/`).
